# Products table: cleaning

Run from the folder that contains raw/ and cleaned/ (your VS Code workspace root).
Needs cleaned/order_items_clean.csv from the order items notebook for the sales-link check.

Decisions and findings for this table are written up in `cleaning_notes/README_products.md`.

### 1. Imports and settings

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 2. Load the raw products file

In [ ]:
products = pd.read_csv(RAW / 'olist_products_dataset.csv')

### 3. First inspection: structure, nulls, preview, key

In [ ]:
products.info()
print(products.isna().sum())          # category/name/description/photos: 610 each; weight and sizes: 2 each
print(products.head())
print(products['product_id'].is_unique)                    # True
print(products['product_category_name'].nunique())         # 73 categories
print(products['product_category_name'].value_counts())    # cama_mesa_banho 3,029 ... cds_dvds_musicais 1

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm       

### 4. Are the 610 missing values the same rows?

In [ ]:
cat_missing = products['product_category_name'].isna()
print(cat_missing.sum())                                                       # 610
print(products.loc[cat_missing, ['product_name_lenght', 'product_description_lenght',
                                 'product_photos_qty']].isna().all().all())    # True: one block of 610 products

610
True


### 5. Products with no weight or size, and the size/weight profile

In [ ]:
print(products[products['product_weight_g'].isna()])       # 2 products
print(products[['product_weight_g', 'product_length_cm',
                'product_height_cm', 'product_width_cm']].describe())
print(products[products['product_weight_g'] == 0])         # 4 products, all cama_mesa_banho, same 30x25x30 size
print((products[['product_length_cm', 'product_height_cm', 'product_width_cm']] == 0).sum())   # all 0

# Are all sizes and weights whole numbers? (needed before converting to Int64)
size_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
print((products[size_cols].dropna() % 1 != 0).sum())       # all 0

# Category text quality
cat = products['product_category_name'].dropna()
print((cat != cat.str.strip()).sum(), (cat != cat.str.lower()).sum())    # 0 0

                             product_id product_category_name  \
8578   09ff539a621711667c43eba6a3bd8466                 bebes   
18851  5eb564652db742ff8f28759cd8d2652a                   NaN   

       product_name_lenght  product_description_lenght  product_photos_qty  \
8578                  60.0                       865.0                 3.0   
18851                  NaN                         NaN                 NaN   

       product_weight_g  product_length_cm  product_height_cm  \
8578                NaN                NaN                NaN   
18851               NaN                NaN                NaN   

       product_width_cm  
8578                NaN  
18851               NaN  
       product_weight_g  product_length_cm  product_height_cm  \
count      32949.000000       32949.000000       32949.000000   
mean        2276.472488          30.815078          16.937661   
std         4282.038731          16.914458          13.637554   
min            0.000000           7

### 6. The translation file

In [ ]:
trans = pd.read_csv(RAW / 'product_category_name_translation.csv')
trans.info()                                               # 71 rows, no nulls
print(trans['product_category_name'].is_unique)            # True
print(set(products['product_category_name'].dropna()) - set(trans['product_category_name']))
# {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}: 2 categories with no English name

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB
True
{'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


### 7. Link with sales

In [ ]:
items = pd.read_csv(CLEAN / 'order_items_clean.csv', usecols=['order_id', 'product_id'])
print((~items['product_id'].isin(products['product_id'])).sum())       # 0 items with an unknown product
print((~products['product_id'].isin(items['product_id'])).sum())       # 0 products that never sold

0
0


### 8. Cleaning step 1: fix the misspelled column names

In [ ]:
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length',
})

### 9. Cleaning step 2: whole numbers, keeping the empty cells

In [ ]:
int_cols = ['product_name_length', 'product_description_length', 'product_photos_qty',
            'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
products[int_cols] = products[int_cols].astype('Int64')
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   product_id                  32951 non-null  str  
 1   product_category_name       32341 non-null  str  
 2   product_name_length         32341 non-null  Int64
 3   product_description_length  32341 non-null  Int64
 4   product_photos_qty          32341 non-null  Int64
 5   product_weight_g            32949 non-null  Int64
 6   product_length_cm           32949 non-null  Int64
 7   product_height_cm           32949 non-null  Int64
 8   product_width_cm            32949 non-null  Int64
dtypes: Int64(7), str(2)
memory usage: 2.5 MB


### 10. Cleaning step 3: English category names (2 missing translations added by hand)

In [ ]:
extra = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc_gamer', 'portable_kitchen_and_food_preparers'],
})
trans_full = pd.concat([trans, extra], ignore_index=True)
products = products.merge(trans_full, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
print(len(products))                                        # still 32,951: a merge must not add rows

32951


### 11. Cleaning step 4: flag weight problems (kept, not changed)

In [ ]:
products['has_size_issue'] = products['product_weight_g'].isna() | (products['product_weight_g'] == 0)
print(products['has_size_issue'].value_counts())            # True 6 / False 32,945

has_size_issue
False    32945
True         6
Name: count, dtype: Int64


### 12. Save the cleaned table and verify the file on disk

In [ ]:
products.to_csv(CLEAN / 'products_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'products_clean.csv')
print(check.shape)                                                       # expect (32951, 11)
print(check['product_id'].is_unique)                                     # True
print(check['has_size_issue'].sum())                                     # 6
print((check['product_category_name_english'] == 'unknown').sum())       # 610
print(check['product_category_name_english'].isna().sum())               # 0
print(check.columns.tolist())                                            # spelling now fixed

(32951, 11)
True
6
610
0
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'has_size_issue']
